# Pipeline del Deliverable 2 — Phi-3.5-mini

Ejecuta las celdas **en orden, de arriba abajo**. No hay que editar ninguna.

Dos mediciones distintas, y conviene correr la primera antes que la segunda:

| sección | qué mide | cuánto tarda |
|---|---|---|
| 6, ablación | solo el paso 3, con cuatro versiones del prompt | ~10 min |
| 7, la grilla | el pipeline completo, 2 variantes × 2 modos | ~25 min |


## 1 · GPU y dependencias
Debe decir `cuda: True` y mostrar una **Tesla T4**.

Si no aparece la T4: menú *Entorno de ejecución* → *Cambiar tipo de entorno de ejecución* →
acelerador **T4 GPU**.


In [ ]:
!nvidia-smi -L
!pip -q install -U transformers accelerate bitsandbytes
import torch; print('cuda:', torch.cuda.is_available())

## 2 · Subir el paquete
Sube `paquete_colab.zip` desde `C:\\Lucas\\Claude\\2026-2\\GenAI\\deliverable-1\\`.

**Ojo:** si ya subiste un archivo con ese nombre en esta sesión, Colab no lo reemplaza, lo
guarda como `paquete_colab (1).zip`. La celda siguiente toma el más reciente, así que no
importa cómo termine llamándose.


In [ ]:
import os
os.chdir('/content')
from google.colab import files
subidos = files.upload()
for nombre, contenido in subidos.items():
    print(f'{nombre}: {len(contenido):,} bytes')

## 3 · Descomprimir y verificar
**Ésta es la celda que decide si se puede seguir.** Corre las autopruebas del repositorio.
Todas tienen que pasar antes de gastar un segundo de GPU.

Debe terminar con `TODO EN ORDEN`.


In [ ]:
import subprocess, sys, os, glob, zipfile

# Colab NO reemplaza un archivo subido que ya existe: lo guarda como "nombre (1).zip".
# Por eso no se busca por nombre, se toma el zip mas reciente de /content.
zips = sorted(glob.glob('/content/*.zip'), key=os.path.getmtime)
if not zips:
    raise SystemExit('no hay ningun .zip en /content: vuelve a la celda anterior')
paquete = zips[-1]
print(f'usando {paquete}  ({os.path.getsize(paquete):,} bytes)')
os.system('unzip -o -q "%s" -d /content/proyecto' % paquete)
os.chdir('/content/proyecto/scripts')
print()

PRUEBAS = [
    ('verificador.py', ['verificador.py']),
    ('ficha.py',       ['ficha.py']),
    ('pipeline.py',    ['pipeline.py']),
    ('runner_d2.py',   ['runner_d2.py', '--pruebas']),
    ('ablacion_p3.py', ['ablacion_p3.py', '--modelo-falso', '--limite', '2']),
]

todo_ok = True
for nombre, args in PRUEBAS:
    if not os.path.exists(args[0]):
        print(f'[FALTA] {nombre:16} no esta en el paquete')
        todo_ok = False
        continue
    r = subprocess.run([sys.executable] + args, capture_output=True, text=True)
    ultima = (r.stdout.strip().splitlines() or ['(sin salida)'])[-1]
    print(f'[{"OK  " if r.returncode == 0 else "FALLA"}] {nombre:16} {ultima[:60]}')
    if r.returncode != 0:
        todo_ok = False
        print(r.stdout[-1200:])
        print(r.stderr[-600:])

print()
print('TODO EN ORDEN' if todo_ok else '*** NO SIGAS: algo fallo ***')

## 4 · Limpiar mediciones anteriores
Si esta sesión ya corrió algo, quedan archivos a medio escribir. El runner los daría por
buenos y los retomaría, mezclando dos mediciones. Esta celda los borra.

Las corridas completas anteriores están guardadas en el repositorio, así que no se pierde nada.


In [ ]:
import glob, os

borrados = glob.glob('/content/proyecto/resultados/*.jsonl')
for f in borrados:
    os.remove(f)
print(f'{len(borrados)} archivos de mediciones anteriores borrados')

## 5 · Modelo
Phi-3.5-mini, 3,8 mil millones de parámetros, cuantizado a 4 bits.


In [ ]:
MODELO = 'microsoft/Phi-3.5-mini-instruct'
print('modelo elegido:', MODELO)

## 6 · Ablación del paso 3
Corre **solo el paso 3**, con la ficha verdadera de cada caso, bajo cuatro versiones del
prompt: `completo`, `sin_reglas`, `sin_ejemplos` y `minimo`.

Sirve para saber qué versión usar antes de gastar 25 minutos midiendo las tres etapas. El
techo demostrado es 60/60. Unos 10 minutos.


In [ ]:
import subprocess, sys, os, time

os.chdir('/content/proyecto/scripts')
t0 = time.time()
r = subprocess.run([sys.executable, 'ablacion_p3.py', '--modelo', MODELO],
                   capture_output=True, text=True)
if r.returncode == 0:
    print(r.stdout[-5000:])
else:
    print('FALLO:')
    print('\n'.join(r.stderr.strip().splitlines()[-12:]))
print('\nablacion completa en %.1f minutos' % ((time.time() - t0) / 60))

## 7 · La grilla
El pipeline completo, cuatro corridas de 60 casos.

| variante | modo | qué mide |
|---|---|---|
| `puro` | `encadenado` | el sistema: el modelo hace los tres pasos |
| `puro` | `oraculo` | la competencia de cada paso por separado |
| `retrieval` | `encadenado` | el sistema con la ficha armada por código |
| `retrieval` | `oraculo` | el paso 3 solo, con ficha perfecta |

Unos 25 minutos. **Si algo se corta, vuelve a correr esta misma celda**: retoma donde iba.


In [ ]:
import subprocess, sys, os, time

os.chdir('/content/proyecto/scripts')
GRILLA = [('puro', 'encadenado'), ('puro', 'oraculo'),
          ('retrieval', 'encadenado'), ('retrieval', 'oraculo')]

t0 = time.time()
for var, modo in GRILLA:
    print('=' * 72)
    print('%s  ·  variante %s  ·  modo %s' % (MODELO, var, modo))
    print('=' * 72)
    cmd = [sys.executable, 'runner_d2.py', '--modelo', MODELO,
           '--variante', var, '--modo', modo]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode == 0:
        print('\n'.join(r.stdout.strip().splitlines()[-16:]))
    else:
        print('FALLO:')
        print('\n'.join(r.stderr.strip().splitlines()[-10:]))
        break
    print()

print('grilla completa en %.1f minutos' % ((time.time() - t0) / 60))

## 8 · Tabla comparativa
El baseline arriba, las corridas del pipeline abajo.


In [ ]:
import json, glob, os

print('{:34} {:>9} {:>8} {:>8} {:>9} {:>8}'.format(
    'condicion', 'decision', 'regla', 'ambas', 'tok/caso', 's/caso'))
print('-' * 82)

f = '/content/proyecto/baseline/Phi-3.5-mini-instruct__few_shot__prosa.raw.jsonl'
if os.path.exists(f):
    r = [json.loads(l) for l in open(f, encoding='utf-8') if l.strip()]
    p = lambda k: 100 * sum(x[k] for x in r) / len(r)
    print('{:34} {:8.1f}% {:7.1f}% {:7.1f}% {:9,.0f} {:8.1f}'.format(
        'baseline few-shot (1 llamada)', p('acierto_decision'), p('acierto_regla'),
        p('acierto_conjunto'),
        sum(x['tokens_entrada'] for x in r) / len(r),
        sum(x['segundos'] for x in r) / len(r)))
else:
    print('  (falta el baseline en baseline/)')

for f in sorted(glob.glob('/content/proyecto/resultados/pipeline__*.raw.jsonl')):
    r = [json.loads(l) for l in open(f, encoding='utf-8') if l.strip()]
    if not r:
        continue
    p = lambda k: 100 * sum(x[k] for x in r) / len(r)
    print('{:34} {:8.1f}% {:7.1f}% {:7.1f}% {:9,.0f} {:8.1f}'.format(
        '%s / %s' % (r[0]['variante'], r[0]['modo']),
        p('acierto_decision'), p('acierto_regla'), p('acierto_conjunto'),
        sum(x['tokens_total'] for x in r) / len(r),
        sum(x['segundos_total'] for x in r) / len(r)))

## 9 · Atribución por paso
Dónde se rompe el sistema.


In [ ]:
import json, glob

for f in sorted(glob.glob('/content/proyecto/resultados/pipeline__*.raw.jsonl')):
    r = [json.loads(l) for l in open(f, encoding='utf-8') if l.strip()]
    if not r:
        continue
    n = len(r)
    print('=' * 72)
    print('variante %s  ·  modo %s   (n=%d)' % (r[0]['variante'], r[0]['modo'], n))
    print('=' * 72)
    print('  paso 1  ramo %5.1f%%   periodo %5.1f%%' % (
        100 * sum(x['paso1']['acierto_ramo'] for x in r) / n,
        100 * sum(x['paso1']['acierto_periodo'] for x in r) / n))
    for c in r[0]['paso2']['acierto']:
        print('  paso 2  %-24s %5.1f%%' % (
            c, 100 * sum(x['paso2']['acierto'][c] for x in r) / n))
    print('  paso 3  decision %5.1f%%   regla %5.1f%%' % (
        100 * sum(x['acierto_decision'] for x in r) / n,
        100 * sum(x['acierto_regla'] for x in r) / n))
    print('  CONJUNTO %5.1f%%' % (100 * sum(x['acierto_conjunto'] for x in r) / n))
    print()

## 10 · Descargar
Baja todo lo medido para comitearlo al repositorio.


In [ ]:
import shutil
shutil.make_archive('/content/resultados_d2', 'zip', '/content/proyecto/resultados')
from google.colab import files
files.download('/content/resultados_d2.zip')